In [1]:
import pandas as pd

df = pd.read_csv("../../datasets/heart.csv")

categorical_columns = ["sex", "cp", "thal"]
categorical_columns = [col for col in categorical_columns if col in df.columns]

print("Dataset shape:", df.shape)
print("Categorical columns:", categorical_columns)

for col in categorical_columns:
    print(f"{col}: {df[col].unique()}")

Dataset shape: (1025, 14)
Categorical columns: ['sex', 'cp', 'thal']
sex: [1 0]
cp: [0 1 2 3]
thal: [3 2 1 0]


In [2]:
def manual_one_hot_encode(values, known_categories=None):
    if known_categories is None:
        known_categories = []

    categories = list(known_categories)

    # Discover categories in the incoming data
    for value in values:
        if pd.isna(value):
            continue

        if value not in categories:
            categories.append(value)

    encoded_rows = []

    for value in values:
        encoded_row = {}

        for category in categories:
            encoded_row[f"category_{category}"] = (
                1 if not pd.isna(value) and value == category else 0
            )

        encoded_row["unknown"] = (
            1 if not pd.isna(value) and value not in known_categories else 0
        )

        encoded_rows.append(encoded_row)

    return pd.DataFrame(encoded_rows, index=values.index), categories


print("Manual one-hot encoding function created.")

Manual one-hot encoding function created.


In [3]:
df_encoded = df.copy()
category_maps = {}

for col in categorical_columns:
    known_categories = sorted(df[col].dropna().unique().tolist(), key=str)

    encoded_data, categories = manual_one_hot_encode(
        df[col],
        known_categories
    )

    category_maps[col] = categories

    encoded_data.columns = [
        f"{col}_{name}" for name in encoded_data.columns
    ]

    df_encoded = pd.concat(
        [df_encoded.drop(columns=[col]), encoded_data],
        axis=1
    )

print("Encoding completed.")
display(df_encoded.head())

Encoding completed.


,age,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,...,cp_category_0,cp_category_1,cp_category_2,cp_category_3,cp_unknown,thal_category_0,thal_category_1,thal_category_2,thal_category_3,thal_unknown
0,52,125,212,0,1,168,0,1.0,2,2,...,1,0,0,0,0,0,0,0,1,0
1,53,140,203,1,0,155,1,3.1,0,0,...,1,0,0,0,0,0,0,0,1,0
2,70,145,174,0,1,125,1,2.6,0,0,...,1,0,0,0,0,0,0,0,1,0
3,61,148,203,0,1,161,0,0.0,2,1,...,1,0,0,0,0,0,0,0,1,0
4,62,138,294,1,1,106,0,1.9,1,3,...,1,0,0,0,0,0,0,1,0,0


In [4]:
test_values = pd.Series([0, 1, 99, None])

known_categories = [0, 1]

test_encoded, discovered_categories = manual_one_hot_encode(
    test_values,
    known_categories
)

print("Test values:")
print(test_values.tolist())

print("\nEncoded output:")
display(test_encoded)

print("\nDiscovered categories:", discovered_categories)

Test values:
[0.0, 1.0, 99.0, nan]

Encoded output:


,category_0,category_1,category_99.0,unknown
0,1,0,0,0
1,0,1,0,0
2,0,0,1,1
3,0,0,0,0



Discovered categories: [0, 1, 99.0]


In [5]:
output_path = "../../datasets/heart_one_hot_encoded.csv"

df_encoded.to_csv(output_path, index=False)

print("Original shape:", df.shape)
print("Encoded shape:", df_encoded.shape)
print("Saved:", output_path)

Original shape: (1025, 14)
Encoded shape: (1025, 24)
Saved: ../../datasets/heart_one_hot_encoded.csv


# Task 8: Manual One-Hot Encoding

## Objective
To implement one-hot encoding using a custom function without built-in encoding utilities, while supporting newly encountered categories.

## Methodology
1. Loaded the Heart Disease dataset.
2. Identified categorical columns.
3. Implemented a custom function to generate binary indicator columns.
4. Reserved an unknown indicator for values outside the known categories.
5. Tested the function using known, unseen, and missing values.
6. Saved the encoded dataset separately.

## Conclusion
A custom one-hot encoding function was implemented and tested. It generates binary columns for categories and identifies unseen values. The original dataset remains unchanged.

When processing future data, the training category list should be retained to distinguish known categories from unseen categories consistently.
